# Collect Replay Buffers

In [1]:
import torch
import numpy as np
from omegaconf import OmegaConf
from functools import partial
import gymnasium as gym
import matplotlib.pyplot as plt
import re
from pathlib import Path
from tqdm.notebook import tqdm

import time

import bbrl_utils
from bbrl_utils.notebook import setup_tensorboard
from bbrl.stats import WelchTTest
from bbrl.agents import Agent, Agents, TemporalAgent
from bbrl.agents.gymnasium import ParallelGymAgent, make_env
from bbrl.workspace import Workspace
from bbrl.utils.replay_buffer import ReplayBuffer

import bbrl_gymnasium

from pmind.algorithms import DQN, DDPG, TD3, OfflineTD3
from pmind.losses import dqn_compute_critic_loss, ddqn_compute_critic_loss

from pmind.replay import (
    collect_policy_transitions,
    collect_uniform_transitions,
    mix_transitions,
    test_rb_uniform_proportions,
)

from pmind.visualization import plot_perf_vs_rb_composition_from_dict

from pmind.config.loader import load_config

bbrl_utils.setup()

%load_ext autoreload
%autoreload 2

KeyboardInterrupt: 

In [ ]:
ENV_NAMES = (
    "CartPoleContinuous-v1",
    "Pendulum-v1",
    "MountainCarContinuous-v0",
    "LunarLanderContinuous-v3",
)

RETRAIN_EXPLOIT = False
COLLECT_BUFFERS = True
ACTION_NOISES = (0., 0.05, 0.1, 0.5, 1., 2.) # TODO: better range ot values?
BUFFER_SIZE = 200_000 

cfg_all = load_config("td3")

Sources of transitions:
- best and intermediate policies with no noise applied
- best and intermediate policies with TruncatedGaussian action noise of various variance
- uniform exploration of state and action space ("teleportation")

In [ ]:
for env_name in ENV_NAMES:
    print("==========" + env_name + "==========")
    cfg = OmegaConf.create(cfg_all.environments[env_name])
    
    # Train exploitation policy (at various levels of performance specified in cfg)
    
    if RETRAIN_EXPLOIT:
        print("Learning policies")
        cfg_online = OmegaConf.create(cfg)
        td3 = TD3(cfg_online)
        td3.train()
        td3.visualize_best()
        learned_policies = td3.get_learned_policies()

        torch.save(learned_policies,f"../models/{env_name}/learned-policies.pt")
    else:
        print("Skipped policy learning")
        learned_policies = torch.load(
            f"../models/{env_name}/learned-policies.pt", weights_only=False
        )
    
    # Collect transitions from policies with and without truncated gaussian noise on actions
    rb_exploit = {}
    pbar = tqdm(learned_policies.items())
    for reward, policy in pbar:
        pbar.set_description(f"Policy ({reward:.2f}) transitions")
        rb_exploit[reward] = {}
        for action_noise in ACTION_NOISES:
            rb_exploit[reward][action_noise] = \
            collect_policy_transitions(policy, 
                                       env_name,
                                       BUFFER_SIZE, 
                                       action_noise=action_noise)
    torch.save(rb_exploit, f"../models/{env_name}/rb-exploit.pt")
    
    # Collect uniform exploration transitions (random actions in random states)
    rb_unif = collect_uniform_transitions(env_name, buffer_size=BUFFER_SIZE)
    torch.save(rb_unif, f"../models/{env_name}/rb-unif.pt")
    

---

# SANDBOX

## Test offline learning on mixed replay buffer

NOTE: here, for **test only**, use the next notebook otherwise

In [ ]:
# PROPORTIONS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
# SEEDS = [0, 1, 2, 3, 4]
# MAX_STEPS = 50_000  # 200_000  # 5000

# cfg_offline = OmegaConf.create(cfg)

# # TODO: integrate it in config files:

# # TODO: do epochs even make sense for offline learning?

# cfg_offline.algorithm.n_steps = MAX_STEPS
# cfg_offline.algorithm.max_epochs = None

# # cfg_offline.algorithm.batch_size = 256

# cfg_offline.algorithm.eval_interval = 100
# cfg_offline.algorithm.nb_evals = 10  # nb of evaluation envs in parallel

# # learning starts immediately for offline:
# cfg_offline.algorithm.learning_starts = None

# # there is no exploration in offline learning
# cfg_offline.algorithm.action_noise = None
# cfg_offline.algorithm.target_policy_noise = None

# rb_exploit_best = rb_exploit[max(rb_exploit)][0] # best reward, 0 noise
# performances = test_rb_uniform_proportions(
#     rb_unif=rb_unif,
#     rb_exploit=rb_exploit_best,
#     buffer_size=BUFFER_SIZE,
#     proportions=PROPORTIONS,
#     agent_constructor=TD3,
#     cfg=cfg_offline,
#     seeds=SEEDS, # TODO: make buffer mixing take this seed as well
# )["performances"]
# torch.save(performances, "performances-temp.pt")
# plot_perf_vs_rb_composition_from_dict(
#     PROPORTIONS,
#     performances,
#     last_n=5,  # NOTE: depends on evaluation interval
#     legend_title="Exploitation Reward",
#     fig_name=f"{ENV_NAME}-offline-diff-best-policies-{int(time.time())}.pdf",
# )
# res = []
# for i, proportion in enumerate(PROPORTIONS):
#     res.append(performances[i].mean(axis=(1, 2)))
# plt.plot(PROPORTIONS, res)
# plt.show()

## Experiment with Gaussian noise around policy

In [ ]:
# from torchrl.modules import TruncatedNormal
# env = gym.make("LunarLanderContinuous-v3")
# action_dim = 1
# mean = torch.tensor([0]*action_dim)
# sigma = torch.tensor([1]*action_dim) # 0.05 0.1 0.5 1 2
# low = torch.tensor([-1]*action_dim)
# high = torch.tensor([1]*action_dim)

# dist = TruncatedNormal(loc=mean,
#                        scale=sigma, 
#                        low=low, 
#                        high=high)

# x = dist.sample((100000,))
# plt.hist(x.squeeze().detach().numpy(), bins=100)
# plt.xlim([float(low)-0.1, float(high)+0.1])
# plt.show()